# 03 Allocate and draw

Nested allocation at `min`, `option`, and `full` under Candidate A (proportional) and Candidate B (tail-boosted floor), then a GRTS draw with oversample, legacy sites, installation order, disturbance and remeasurement flags. The Python GRTS here is for iteration; the frozen design is drawn with `scripts/grts_draw.R` (spsurvey) from the files this notebook exports.

In [ ]:
import sys, os
print(sys.executable)
from pathlib import Path
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd
from src.io import load_config, get_logger
from src import strata, qa
cfg = load_config()
log = get_logger("03_allocate_draw")
log.info(f"Project: {cfg['project']['name']} | synthetic={cfg['run']['synthetic']} | freeze={cfg['run']['freeze']}")
P = Path(cfg["paths"]["processed"]); O = Path(cfg["paths"]["outputs"]); P.mkdir(parents=True, exist_ok=True); O.mkdir(exist_ok=True)

In [ ]:
frame = pd.read_parquet(P / "frame_classified.parquet")
cells = pd.read_csv(O / "cells.csv")

## Allocation at three nested levels

In [ ]:
alloc = pd.concat([strata.allocate(cells, cfg, lvl) for lvl in cfg["allocation"]["levels"]], ignore_index=True)
alloc = strata.apply_nevada_floor(alloc, frame.groupby(["cell_id", "state"], as_index=False)["acres"].sum(), cfg)
alloc.to_csv(O / "allocation_table.csv", index=False)
for lvl in cfg["allocation"]["levels"]:
    a = alloc[alloc["level"] == lvl]
    log.info(f"{lvl}: n={a['n_B'].sum()}  cells at floor (B)={int((a['n_B'] == cfg['allocation']['floor_per_cell'][lvl]).sum())}  cells with 0-1 plots (A)={int((a['n_A'] <= 1).sum())}")
alloc.pivot_table(index=["forest_type", "seral_class", "density_class"], columns="level", values=["n_A", "n_B"]).astype(int)

## Inclusion weights: disturbance targeting and tail boost

Disturbance is not a stratum. It enters as an inclusion weight that lifts post-fire and scheduled-treatment units so the expected share of the sample lands near `allocation.disturbance_shares`. Representativeness weighting is switched on with `draw.weight_by_representativeness` once Shengli's raster is in hand.

In [ ]:
ds = cfg["allocation"]["disturbance_shares"]
w = pd.Series(1.0, index=frame.index)
for flag, share in [("post_fire", ds["post_fire"]), ("treatment_2027_2031", ds["treatment"])]:
    p_frame = frame[flag].mean()
    if 0 < p_frame < share:
        w[frame[flag]] *= share / p_frame          # lift so expected share ~ target
    log.info(f"{flag}: frame share {p_frame:.3f}, target {share:.2f}")
if cfg["draw"]["weight_by_representativeness"] and "representativeness" in frame:
    w *= 1 + (frame["representativeness"].rank(pct=True))   # later-imputed (less represented) units up-weighted
frame["inclusion_weight"] = w

## Legacy sites

Existing permanently marked plots that fall inside the frame are attached to the sampling unit that contains them and passed to the draw as legacy sites. They count toward the cell's allocation, the new plots balance around them, and the minimum-distance rule keeps new plots off them.

In [ ]:
legacy_mask = pd.Series(False, index=frame.index)
if not cfg["run"]["synthetic"]:
    import geopandas as gpd
    from shapely.geometry import Point
    pts = gpd.GeoDataFrame(frame, geometry=[Point(xy) for xy in zip(frame.x, frame.y)], crs=cfg["crs"]["working"])
    from src.layers import read_layer
    for key in cfg["draw"]["legacy_sources"]:
        src_k = cfg["sources"][key]
        if not (str(src_k).startswith("http") or Path(src_k).exists()): log.info(f"{key}: not available yet"); continue
        leg = read_layer(src_k, cfg, log=log)
        half_diag = cfg["crs"]["lidar_grid_m"] * cfg["frame"]["unit_px"] * 0.7072   # any point in the block is this close to its centre
        near = gpd.sjoin_nearest(leg[["geometry"]], pts[["unit_id", "geometry"]], max_distance=half_diag)
        legacy_mask[frame["unit_id"].isin(near["unit_id"])] = True
        log.info(f"{key}: {len(leg)} sites, {near['unit_id'].nunique()} inside the frame")
else:
    legacy_mask[frame.sample(12, random_state=1).index] = True   # pretend a dozen LTW plots exist
frame["legacy"] = legacy_mask
log.info(f"Legacy sites in frame: {int(legacy_mask.sum())}")

## GRTS draw (Python, for iteration)

One unit per site, and no two sites closer than `draw.min_distance_m` (two macroplot radii), legacy sites included, so no plot footprints overlap. The frozen draw applies the same rule through spsurvey's `mindis`.

In [ ]:
full = alloc[alloc["level"] == "full"].set_index("cell_id")["n_B"].to_dict()
sel = strata.grts_draw(frame, full, frame["inclusion_weight"], seed=cfg["draw"]["seed"],
                       oversample_factor=cfg["draw"]["oversample_factor"], legacy_mask=frame["legacy"],
                       min_distance_m=cfg["draw"]["min_distance_m"])
sel = strata.installation_order(sel, cfg, alloc).merge(
    sel[sel["status"] == "backup"], how="outer")
sel["status"] = sel["status"].fillna("backup")
rng = np.random.default_rng(cfg["draw"]["seed"] + 1)
prim = sel["status"].isin(["primary", "legacy"]) & (sel["level"] == "min")
sel["remeasure_flag"] = False
sel.loc[sel[prim].sample(frac=cfg["draw"]["remeasure_share"], random_state=int(rng.integers(1e6))).index, "remeasure_flag"] = True
sel["plot_id"] = "FH-" + sel["cell_id"] + "-" + sel.groupby("cell_id").cumcount().add(1).astype(str).str.zfill(3)
log.info(f"Selected: {len(sel)} rows; primary {int((sel.status=='primary').sum())}, legacy {int((sel.status=='legacy').sum())}, backup {int((sel.status=='backup').sum())}")
sel.groupby(["level", "status"]).size().unstack(fill_value=0)

## Export for the frozen spsurvey draw and for review

In [ ]:
alloc[alloc["level"] == "full"][["cell_id", "n_B"]].to_csv(P / "allocation_full.csv", index=False)
try:
    import geopandas as gpd
    from shapely.geometry import Point
    keep_cols = [c for c in ["unit_id", "cell_id", "forest_type", "inclusion_weight", "balance_caty", "x", "y"] if c in frame]
    gpd.GeoDataFrame(frame[keep_cols],
                     geometry=[Point(xy) for xy in zip(frame.x, frame.y)], crs=cfg["crs"]["working"]).to_file(P / "frame_points.gpkg", driver="GPKG")
    gpd.GeoDataFrame(frame.loc[frame["legacy"], [c for c in ["unit_id", "cell_id", "forest_type"] if c in frame]],
                     geometry=[Point(xy) for xy in zip(frame.loc[frame.legacy].x, frame.loc[frame.legacy].y)], crs=cfg["crs"]["working"]).to_file(P / "legacy_sites.gpkg", driver="GPKG")
    stamp = pd.Timestamp.today().strftime("%Y%m%d")
    internal = gpd.GeoDataFrame(sel, geometry=[Point(xy) for xy in zip(sel.x, sel.y)], crs=cfg["crs"]["working"])
    internal.to_file(O / f"plots_design_{stamp}_internal.gpkg", driver="GPKG")
    internal.drop(columns=cfg["run"]["publish_strip_fields"]).to_file(O / f"plots_design_{stamp}.gpkg", driver="GPKG")
    log.info(f"Wrote plots_design_{stamp}.gpkg (published) and _internal.gpkg (with remeasure flag)")
except Exception as e:
    log.warning(f"GeoPackage export skipped: {e}")
sel.to_parquet(P / "selected.parquet", index=False)
sel.to_csv(O / "plot_list_review.csv", index=False)
if cfg["run"]["freeze"]:
    log.info(f"FROZEN design: seed={cfg['draw']['seed']} levels={cfg['allocation']['levels']}")